In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0"
    !pip install --no-deps unsloth
!pip install protobuf==3.20.3
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install --no-deps transformers-cfg
!pip install --upgrade --quiet peft

In [2]:
SEED = 42
MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/Qwen2.5-Coder-3B"

In [3]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    dtype=None,
    load_in_4bit=True,
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-01-02 16:11:48.646142: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767370309.003677      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767370309.108250      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.10: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
import json

dataset = []
with open("/kaggle/input/dataset/dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        dataset.append(json.loads(line))

In [5]:
display(dataset[0])

{'messages': [{'role': 'system',
   'content': 'You are a text-to-SQL assistant. Generate valid SQL syntax for DuckDB. Never explain the query or use markdown. Limit the query to SELECT statements only. Avoid making any assumptions about schema. '},
  {'role': 'user',
   'content': 'Given the schema: [{"table": "perpetrator", "columns": ["Perpetrator_ID", "People_ID", "Date", "Year", "Location", "Country", "Killed", "Injured"]}, {"table": "people", "columns": ["People_ID", "Name", "Height", "Weight", "Home Town"]}]. Answer the question: How many perpetrators are there?'},
  {'role': 'assistant', 'content': 'SELECT count(*) FROM perpetrator;'}]}

In [6]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        texts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        )
    return {"text": texts}

dataset = Dataset.from_list(dataset)
dataset = dataset.map(formatting_prompts_func, batched=True)
dataset = standardize_sharegpt(dataset)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [7]:
dataset[0]["messages"]

[{'content': 'You are a text-to-SQL assistant. Generate valid SQL syntax for DuckDB. Never explain the query or use markdown. Limit the query to SELECT statements only. Avoid making any assumptions about schema. ',
  'role': 'system'},
 {'content': 'Given the schema: [{"table": "perpetrator", "columns": ["Perpetrator_ID", "People_ID", "Date", "Year", "Location", "Country", "Killed", "Injured"]}, {"table": "people", "columns": ["People_ID", "Name", "Height", "Weight", "Home Town"]}]. Answer the question: How many perpetrators are there?',
  'role': 'user'},
 {'content': 'SELECT count(*) FROM perpetrator;', 'role': 'assistant'}]

In [8]:
dataset[0]["text"]

'<|im_start|>system\nYou are a text-to-SQL assistant. Generate valid SQL syntax for DuckDB. Never explain the query or use markdown. Limit the query to SELECT statements only. Avoid making any assumptions about schema. <|im_end|>\n<|im_start|>user\nGiven the schema: [{"table": "perpetrator", "columns": ["Perpetrator_ID", "People_ID", "Date", "Year", "Location", "Country", "Killed", "Injured"]}, {"table": "people", "columns": ["People_ID", "Name", "Height", "Weight", "Home Town"]}]. Answer the question: How many perpetrators are there?<|im_end|>\n<|im_start|>assistant\nSELECT count(*) FROM perpetrator;<|im_end|>\n'

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    bias="none",
    lora_alpha=16,
    lora_dropout=0,
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
    use_gradient_checkpointing="unsloth",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",]
)

Unsloth 2025.12.10 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [10]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args = SFTConfig(
        seed=SEED,
        max_steps=100,
        warmup_steps=5,
        logging_steps=1,
        num_train_epochs=1,
        learning_rate=2e-4,
        weight_decay=0.001,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        lr_scheduler_type="linear",
        optim="paged_adamw_8bit",
        output_dir="outputs",
        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/7000 [00:00<?, ? examples/s]

In [11]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,000 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss
1,2.484700
2,2.008200
3,1.968400
4,2.151600
5,1.833800
6,0.813400
7,1.999100
8,0.985900
9,1.471700
10,1.337000


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=100, training_loss=0.8086889424920082, metrics={'train_runtime': 281.7462, 'train_samples_per_second': 1.42, 'train_steps_per_second': 0.355, 'total_flos': 2145281369284608.0, 'train_loss': 0.8086889424920082, 'epoch': 0.05714285714285714})

In [12]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a DuckDB text-to-SQL assistant. "
            "Generate valid SQL syntax for DuckDB. "
            "Never explain the query or use markdown. "
            "Limit the query to SELECT statements only. "
            "Avoid making any assumptions about schema."
        ),
    },
    {
        "role": "user",
        "content": 'Given the schema: '
        '[{"table": "employee", "columns": ["Employee_ID", "Name", "Department", "Salary"]},'
        '{"table": "project", "columns": ["Project_ID", "Project_Name", "Budget"]},'
        '{"table": "assignment", "columns": ["Employee_ID", "Project_ID", "Hours"]}]. '
        'Answer the question: Which departments have employees assigned to projects with total hours exceeding 100?'
    },
]

In [13]:
from transformers import TextStreamer, StoppingCriteriaList, StoppingCriteria
from unsloth.chat_templates import get_chat_template

class StopOnSemicolon(StoppingCriteria):
    def __call__(self, input_ids, scores, **kwargs):
        last_token_id = input_ids[0, -1].item()
        semicolon_id = tokenizer.encode(";")[-1]
        return last_token_id == semicolon_id

stopping_criteria = StoppingCriteriaList([StopOnSemicolon()])

input_ids = tokenizer.apply_chat_template(
    messages,
    padding=True,
    tokenize=True,
    add_generation_prompt=True,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

print(tokenizer.decode(input_ids[0]))

<|im_start|>system
You are a DuckDB text-to-SQL assistant. Generate valid SQL syntax for DuckDB. Never explain the query or use markdown. Limit the query to SELECT statements only. Avoid making any assumptions about schema.<|im_end|>
<|im_start|>user
Given the schema: [{"table": "employee", "columns": ["Employee_ID", "Name", "Department", "Salary"]},{"table": "project", "columns": ["Project_ID", "Project_Name", "Budget"]},{"table": "assignment", "columns": ["Employee_ID", "Project_ID", "Hours"]}]. Answer the question: Which departments have employees assigned to projects with total hours exceeding 100?<|im_end|>
<|im_start|>assistant



In [14]:
output_ids = model.generate(
    input_ids=input_ids,
    max_new_tokens=128,
    temperature=0.0,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
    stopping_criteria=stopping_criteria,
)

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


system
You are a DuckDB text-to-SQL assistant. Generate valid SQL syntax for DuckDB. Never explain the query or use markdown. Limit the query to SELECT statements only. Avoid making any assumptions about schema.
user
Given the schema: [{"table": "employee", "columns": ["Employee_ID", "Name", "Department", "Salary"]},{"table": "project", "columns": ["Project_ID", "Project_Name", "Budget"]},{"table": "assignment", "columns": ["Employee_ID", "Project_ID", "Hours"]}]. Answer the question: Which departments have employees assigned to projects with total hours exceeding 100?
assistant
SELECT T1.department FROM employee AS T1 JOIN assignment AS T2 ON T1.employee_id  =  T2.employee_id JOIN project AS T3 ON T2.project_id  =  T3.project_id GROUP BY T1.department HAVING sum(T2.hours)  >  100;


In [17]:
from huggingface_hub import login
login()

In [18]:
model.push_to_hub_gguf(
    "thisisdilmurod/qwen2.5-coder-3b-lora-q4km",
    tokenizer,
    quantization_method="q4_k_m"
)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

...

Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_v3jgxs4n`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen2.5-coder-3b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['qwen2.5-coder-3b.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/qwen2.5-coder-3b'. Ski

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading config.json...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/thisisdilmurod/qwen2.5-coder-3b-lora-q4km
Unsloth: Cleaning up temporary files...


'thisisdilmurod/qwen2.5-coder-3b-lora-q4km'